# Preparing data for fire impact analysis

This notebook demonstrates the process for processing input data for use with the fire impacts library.

The process is largely automated and, in many cases, the data is downloaded automatically from the publisher.

There are a few things that you will need, however:

* A catchment boundary for your area of interest (eg Shapefile, GeoJSON)
* A suitable, high resolution DEM for the area. The 1" SRTM derived DEM-H is a good option for most cases.
* The date of a fire event within the catchment,
* Several other datasets that you will need to manually download and store locally


## Installation

You will need a Python installation with the various libaries installed.

As a starting point, a base scientific Python installation that includes `numpy`, `pandas`, `jupyter`, `matplotlib`. For Windows users, the easiest way to set up such an environment is to use Anaconda Python, or miniconda.

This base environment should be extended with specific libraries that are used by the fire impacts library. These are listed in `requirements.txt` and can be installed using `pip` from a command prompt:

```
cd <directory-with-library>
pip install -r requirements.txt
```

Finally, the fire impacts library itself should be installed. If you have cloned the git repository, you can install from your local copy. From the command prompt:

```
cd <directory-with-library>
pip install -e .
```

When the installation has completed, the following import statements should run without error

In [3]:
from fire_impacts import FireImpactsProject

In [4]:
from fire_impacts.pre import topography, severity, soil, rusle

## Logging

We use logging statements to provide feedback on progress through various steps. This allows you to tailor what level of information you see by setting a log 'level':

* `DEBUG`: Low level information about progress
* `INFO`: General progress updates
* `WARNING`: Problems or potential problems that the system can handle
* `EROR`: Problems that prevent the system from running correctly

When you set a log level, you will see those messages as well as the more serious ones. So if you choose `INFO`, you will also see `WARNING` and `ERROR` messages. If you choose `WARNING`, you will also see `ERROR`.

In [6]:
import logging
logging.basicConfig(level=logging.INFO,format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

## Projects

We organise all the data related to a study in a project directory. This should generally be a *new* directory that will be populated with data that has been computed with the library. The library will manage the subdirectories and data within the main project directory.

In Python, we use a `FireImpactsProject` object to represent this directory and all the data stored within it

Here, we create a new project in the current directory. **Note:** In this case we will delete (`clear`) any existing data in that directory.

In [7]:
proj = FireImpactsProject('./fire-impacts-data',clear=True)

2024-11-08 13:22:30,090 - fire_impacts.pre.project - INFO - Clearing existing project folder: ./fire-impacts-data


## Pro processing steps

We will now work through each step of data pre-processing to support for the fire impacts modelling:

* **Topography**: Determine various topographic properties from a DEM, including headwater catchments used to model debris flow triggers
* **Fire severity**: Analyse satellite information to determine the intensity of a historical fire event
* **Soils**: Extract relevant soil properties from the Soil and Landscape Grid of Australia
* **RUSLE**: Compute RUSLE terms (K, L, S and C), including fire modified versions of K and C

The following sections describe each process and the data required.